In [2]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpz7ltstq1".


In [3]:
%%cuda

#include <iostream>
using namespace std;

__global__ void vectorAdd(int *a,int *b,int *c)
{
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

int main()
{
    int n = 5;
    int size = n * sizeof(int);

    int a[] = {1,2,3,4,5};
    int b[] = {10,20,30,40,50};
    int c[5];

    int *d_a,*d_b,*d_c;

    cudaMalloc(&d_a,size);
    cudaMalloc(&d_b,size);
    cudaMalloc(&d_c,size);

    cudaMemcpy(d_a,a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,b,size,cudaMemcpyHostToDevice);

    vectorAdd<<<1,n>>>(d_a,d_b,d_c);

    cudaMemcpy(c,d_c,size,cudaMemcpyDeviceToHost);

    cout<<"Vector Addition Result:\n";

    for(int i=0;i<n;i++)
        cout<<c[i]<<" ";

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Vector Addition Result:
11 22 33 44 55 


In [4]:
%%cuda

#include <iostream>
using namespace std;

#define N 2

__global__ void matrixMul(int *A,int *B,int *C)
{
    int row = threadIdx.y;
    int col = threadIdx.x;

    int sum = 0;

    for(int k=0;k<N;k++)
    {
        sum += A[row*N+k] * B[k*N+col];
    }

    C[row*N+col] = sum;
}

int main()
{
    int A[N][N] = {{1,2},{3,4}};
    int B[N][N] = {{5,6},{7,8}};
    int C[N][N];

    int size = N * N * sizeof(int);

    int *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    cudaMemcpy(d_A,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,size,cudaMemcpyHostToDevice);

    dim3 threads(N,N);

    matrixMul<<<1,threads>>>(d_A,d_B,d_C);

    cudaMemcpy(C,d_C,size,cudaMemcpyDeviceToHost);

    cout<<"Matrix Multiplication Result:\n";

    for(int i=0;i<N;i++)
    {
        for(int j=0;j<N;j++)
        {
            cout<<C[i][j]<<" ";
        }
        cout<<endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Matrix Multiplication Result:
19 22 
43 50 

